In [ ]:
# ! pip install healpy numpyro corner arviz optax reproject einops seaborn chainconsumer blackjax

In [ ]:
! nvidia-smi

In [4]:
import sys
import pickle
sys.path.append("../")

from jax.config import config
config.update("jax_enable_x64", True)

import jax
import jax.numpy as jnp
import numpy as np
import corner
import matplotlib.pyplot as plt

from models.np_model import NPModel

%load_ext autoreload
%autoreload 2

In [5]:
r_outer = 25
l_max = 0

vary_disk = 1
vary_gamma = 1
bulge_hybrid = 1

dif = "ModelO"
ps_cat = "3fgl"
nside = 128

npmodel = NPModel(dif=dif, r_outer=r_outer, l_max=l_max, vary_disk=vary_disk, vary_gamma=vary_gamma, bulge_hybrid=bulge_hybrid, ps_cat=ps_cat, nside=nside)
# svi_results = npmodel.fit_svi(rng_key=jax.random.PRNGKey(4234), n_steps=5000)

# arviz_post = npmodel.get_posterior_samples(rng_key=jax.random.PRNGKey(42342), num_samples=50000)
# posterior = arviz_post.to_dict()['posterior']
# posterior['logZ'] = np.array(jnp.mean(svi_results.losses[-250:]))  # Save log-evidence estimate

posterior_file = "../data/posteriors/posterior_dif_{}_r_{}_lmax_{}_vary_disk_{}_bulge_hybrid_{}_vary_gamma_{})_{}_nside_{}.p".format(dif, r_outer, l_max, vary_disk, bulge_hybrid, vary_gamma, ps_cat, nside)

# # Save data (serialize)
# with open(posterior_file, 'wb') as outfile:
#     pickle.dump(posterior, outfile, protocol=pickle.HIGHEST_PROTOCOL)

# Load data (deserialize)
with open(posterior_file, 'rb') as handle:
    posterior = pickle.load(handle)

Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


## NUTS

In [6]:
from numpyro.infer import MCMC, HMC, HMCECS, NUTS, Predictive

In [7]:
# svi_results = npmodel.fit_svi(rng_key=jax.random.PRNGKey(489234), n_steps=2500)
# arviz_post = npmodel.get_posterior_samples(rng_key=jax.random.PRNGKey(42342), num_samples=5000)
# posterior = arviz_post.to_dict()['posterior']
posterior.pop('logZ', None)
initial_position = {}
for key in posterior.keys():
    initial_position[key] = jnp.mean(posterior[key])

In [8]:
inverse_mass_matrix = {}
for key in posterior.keys():
    inverse_mass_matrix[key] = jnp.var(posterior[key])

In [9]:
inverse_mass_matrix = np.cov(np.array([posterior[key] for key in posterior.keys()])[:, 0, :].T, rowvar=0)

In [10]:
# from numpyro.infer.reparam import NeuTraReparam

# neutra = NeuTraReparam(npmodel.guide, svi_results.params)
# neutra_model = neutra.reparam(npmodel.model)

# kernel = NUTS(neutra_model, max_tree_depth=6, dense_mass=True, inverse_mass_matrix=inverse_mass_matrix, step_size=0.1)
# num_samples = 5000
# mcmc = MCMC(kernel, num_warmup=50, num_samples=num_samples)
# rng_key = jax.random.PRNGKey(0)
# mcmc.run(rng_key, data=npmodel.data, init_params=initial_position)

In [11]:
num_chains = 4

initial_position_batch = {}
for key in posterior.keys():
    initial_position_batch[key] = (1 + 0.2 * np.random.randn(num_chains)) * jnp.mean(posterior[key])

In [12]:
kernel = NUTS(npmodel.model, max_tree_depth=3, dense_mass=False, inverse_mass_matrix=inverse_mass_matrix, step_size=0.1)
num_samples = 10000
mcmc = MCMC(kernel, num_warmup=100, num_samples=num_samples, num_chains=num_chains, chain_method='vectorized')
rng_key = jax.random.PRNGKey(0)
mcmc.run(rng_key, data=npmodel.data, init_params=initial_position_batch)

sample: 100%|██████████| 10100/10100 [2:32:14<00:00,  1.11it/s] 


In [13]:
mcmc.print_summary()


                     mean       std    median      5.0%     95.0%     n_eff     r_hat
              C      1.88      0.83      1.86      0.50      3.25    166.68      1.01
          S_bub      1.23      0.11      1.24      1.06      1.40     27.19      1.12
          S_dif     11.17      0.15     11.17     10.93     11.41    705.38      1.01
          S_gce      0.70      0.50      0.62      0.01      1.43      8.55      1.39
          S_ics      4.84      0.15      4.89      4.62      5.00    103.50      1.02
          S_iso      0.65      0.26      0.64      0.21      1.06     53.34      1.09
          S_psc      2.29      1.39      2.27      0.02      4.25    130.47      1.03
        Sps_dsk      2.10      0.38      2.12      1.46      2.72     24.58      1.15
        Sps_gce      0.74      0.38      0.70      0.18      1.33     20.17      1.21
  f_bulge_poiss      0.34      0.23      0.30      0.00      0.67      7.87      1.32
     f_bulge_ps      0.57      0.24      0.58      0.

In [21]:
az.from_numpyro(mcmc)

Inference data with groups:
	> posterior
	> log_likelihood
	> sample_stats
	> observed_data

In [20]:
import arviz as az
az.summary(az.from_dict(posterior))

Shape validation failed: input_shape: (1, 50000), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
C,2.035,0.681,0.891,3.313,0.003,0.002,50125.0,49064.0,NaN
S_bub,1.203,0.085,1.048,1.368,0.000,0.000,49124.0,47723.0,NaN
S_dif,11.207,0.144,10.940,11.479,0.001,0.000,49127.0,49263.0,NaN
S_gce,0.969,0.250,0.527,1.451,0.001,0.001,49013.0,46088.0,NaN
S_ics,4.873,0.096,4.707,4.987,0.000,0.000,49298.0,48383.0,NaN
S_iso,0.720,0.204,0.363,1.103,0.001,0.001,49827.0,49164.0,NaN
S_psc,2.271,1.412,0.053,4.564,0.006,0.004,49698.0,49878.0,NaN
Sps_dsk,1.907,0.203,1.529,2.281,0.001,0.001,49220.0,48805.0,NaN
Sps_gce,0.590,0.166,0.294,0.900,0.001,0.001,49491.0,49630.0,NaN
f_bulge_poiss,0.243,0.173,0.010,0.572,0.001,0.001,50010.0,50154.0,NaN


In [16]:
arviz_post_mcmc = az.from_dict(mcmc.get_samples())

In [17]:
az.summary(arviz_post_mcmc)

Shape validation failed: input_shape: (1, 40000), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
C,1.879,0.828,0.108,3.235,0.063,0.045,158.0,107.0,NaN
S_bub,1.231,0.106,1.025,1.421,0.017,0.012,38.0,129.0,NaN
S_dif,11.167,0.147,10.890,11.442,0.005,0.004,736.0,2125.0,NaN
S_gce,0.699,0.501,0.011,1.570,0.232,0.175,4.0,21.0,NaN
S_ics,4.839,0.154,4.546,5.000,0.016,0.011,94.0,219.0,NaN
S_iso,0.647,0.261,0.178,1.140,0.040,0.028,43.0,263.0,NaN
S_psc,2.288,1.385,0.017,4.533,0.117,0.083,135.0,182.0,NaN
Sps_dsk,2.095,0.381,1.389,2.833,0.070,0.050,32.0,89.0,NaN
Sps_gce,0.743,0.381,0.110,1.384,0.091,0.066,20.0,67.0,NaN
f_bulge_poiss,0.335,0.235,0.000,0.722,0.128,0.100,4.0,28.0,NaN


In [59]:
# from numpyro.contrib.nested_sampling import NestedSampler

In [58]:
# ns = NestedSampler(npmodel.model)
# ns.run(jax.random.PRNGKey(0), data=npmodel.data)

## BlackJax stuff

In [44]:
from tqdm import tqdm
from typing import Any, Callable, Iterable, Tuple, Union
from jax.experimental import host_callback

def progress_bar_scan(num_samples: int, message: Union[None, str] = None) -> Callable:
    """Decorator factory to build a tqdm progress bar using lax.scan.
    This returns a decorator to apply to the 'body' function used in lax.scan
    Args:
        num_samples (int): number of samples to run lax.scan
        message (Union[None, str]): message to display in the progress bar. Defaults to f'Running for {num_samples:,} iterations'
    Returns:
        Callable: decorator to apply to the body function used in lax.scan
    """

    if message is None:
        message = f"Running for {num_samples:,} iterations"
    tqdm_bars = {}

    def _define_tqdm(arg, transform):
        tqdm_bars[0] = tqdm(range(num_samples))
        tqdm_bars[0].set_description(message, refresh=False)

    def _update_tqdm(arg, transform):
        tqdm_bars[0].update(arg)

    def _close_tqdm(arg, transform):
        tqdm_bars[0].close()

    def _update_progress_bar(iter_num, print_rate):
        "Updates tqdm progress bar of a JAX scan or loop"
        _ = jax.lax.cond(
            iter_num == 0,
            lambda _: host_callback.id_tap(_define_tqdm, print_rate, result=iter_num),
            lambda _: iter_num,
            operand=None,
        )

        _ = jax.lax.cond(
            iter_num % print_rate == 0,
            lambda _: host_callback.id_tap(_update_tqdm, print_rate, result=iter_num),
            lambda _: iter_num,
            operand=None,
        )

        _ = jax.lax.cond(
            iter_num == num_samples - 1,
            lambda _: host_callback.id_tap(_close_tqdm, print_rate, result=iter_num),
            lambda _: iter_num,
            operand=None,
        )

    def _progress_bar_scan(func):
        """Decorator that adds a progress bar to `body_fun` used in `lax.scan`.
        Note that `body_fun` must either be looping over `np.arange(num_samples)`,
        or be looping over a tuple who's first element is `np.arange(num_samples)`
        This means that `iter_num` is the current iteration number
        """
        print_rate = int(num_samples / 20)

        def wrapper_progress_bar(carry, x):
            if type(x) is tuple:
                iter_num, *_ = x
            else:
                iter_num = x
            _update_progress_bar(iter_num, print_rate)
            return func(carry, x)

        return wrapper_progress_bar

    return _progress_bar_scan

## MEADS

In [53]:
# import blackjax
# from numpyro.infer.util import initialize_model

# rng_key = jax.random.PRNGKey(0)
# init_params, potential_fn_gen, *_ = initialize_model(
#     rng_key,
#     npmodel.model,
#     model_args=(npmodel.data,),
#     dynamic_args=True,
# )

# logprob_fn = lambda position: -potential_fn_gen(npmodel.data,)(position)

In [54]:
# [n_chain, n_warm, n_iter] = [64, 100, 100]
# ksam, kinit = jax.random.split(jax.random.PRNGKey(0), 2)

# initial_position_batch = {}
# for key in posterior.keys():
#     initial_position_batch[key] = (1 + 0.2 * np.random.randn(n_chain)) * jnp.mean(posterior[key])
    
# k_warm, k_sample = jax.random.split(ksam)
# warmup = blackjax.meads(logprob_fn, num_chain=n_chain, num_steps=n_warm, batch_fn=jax.vmap)

# chain_state, kernel, _ = warmup.run(k_warm, initial_position_batch)

# # init_state = chain_state.states


# # def one_chain(k_sam, init_state):
# #     state, info = inference_loop(k_sam, init_state, kernel, n_iter)
# #     return state.position, info


# # k_sample = jrnd.split(k_sample, n_chain)
# # samples, infos = batch_fn(one_chain)(k_sample, init_state)
# # tic2 = pd.Timestamp.now()


In [42]:
def inference_loop(rng_key, kernel, initial_state, num_samples, pbar=True):
    
    def one_step(carry, i):
        state, key = carry
        kernel_key, key = jax.random.split(key)
        state, _ = kernel(kernel_key, state)
        return (state, key), state

    lebody = progress_bar_scan(num_samples)(one_step) if pbar else one_step
    _, states = jax.lax.scan(lebody, (initial_state, rng_key), jnp.arange(num_samples))
    return states


In [43]:
import blackjax
from numpyro.infer.util import initialize_model

rng_key = jax.random.PRNGKey(0)
init_params, potential_fn_gen, *_ = initialize_model(
    rng_key,
    npmodel.model,
    model_args=(npmodel.data,),
    dynamic_args=True,
)

logprob_fn = lambda position: -potential_fn_gen(npmodel.data,)(position)

In [13]:
svi_results = npmodel.fit_svi(rng_key=jax.random.PRNGKey(4234), n_steps=1000)
arviz_post = npmodel.get_posterior_samples(rng_key=jax.random.PRNGKey(42342), num_samples=5000)
posterior = arviz_post.to_dict()['posterior']
initial_position = {}
for key in posterior.keys():
    initial_position[key] = jnp.mean(posterior[key])

100%|██████████| 1000/1000 [03:20<00:00,  4.99it/s, init loss: 26055.4020, avg. loss [951-1000]: 20092.8584]


In [ ]:
num_warmup = 200

adapt = blackjax.window_adaptation(
    blackjax.nuts, logprob_fn, num_warmup, target_acceptance_rate=0.8, progress_bar=True
)

last_state, kernel, _ = adapt.run(rng_key, initial_position)

Running window adaptation


In [ ]:
num_sample = 1000

states = inference_loop(rng_key, kernel, last_state, num_sample)

  0%|          | 0/1000 [00:00<?, ?it/s]

## MALA

In [7]:
npmodel = NPModel(dif=dif, r_outer=r_outer, l_max=l_max, vary_disk=False, vary_gamma=False, bulge_hybrid=False, ps_cat=ps_cat, nside=nside)

Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


In [8]:
import blackjax
from numpyro.infer.util import initialize_model

rng_key = jax.random.PRNGKey(0)

init_params, potential_fn_gen, *_ = initialize_model(
    rng_key,
    npmodel.model,
    model_args=(npmodel.data,),
    dynamic_args=True,
)

logprob_fn = lambda position: -potential_fn_gen(npmodel.data,)(position)

In [23]:
initial_position_batch = {}
for key in posterior.keys():
    initial_position_batch[key] = jnp.ones(1) * jnp.mean(posterior[key])

In [24]:
for key in ['C', 'zs', 'gamma_poiss', 'gamma_ps', 'f_bulge_ps', 'f_bulge_poiss']:
    initial_position_batch.pop(key, None)

In [25]:
logprob_fn_vmap = jax.vmap(logprob_fn)

In [26]:
# logprob_fn_vmap(initial_position_batch)

In [27]:
warmup = blackjax.meads(logprob_fn, num_chain=1, num_steps=10)
last_states, kernel, _ = warmup.run(
            jax.random.PRNGKey(0),
            initial_position_batch,
        )

ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1572864 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    3.00MiB
              constant allocation:         0B
        maybe_live_out allocation:    1.50MiB
     preallocated temp allocation:         0B
                 total allocation:    4.50MiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 1.50MiB
		Entry Parameter Subshape: f64[196608]
		==========================

	Buffer 2:
		Size: 1.50MiB
		Entry Parameter Subshape: f64[196608]
		==========================

	Buffer 3:
		Size: 1.50MiB
		Operator: op_name="jit(fn)/jit(main)/mul" source_file="../models/np_model.py" source_line=139
		XLA Label: multiply
		Shape: f64[196608]
		==========================

